1\. Environment Setup
---------------------

The authors utilized a Linux-based server with NVIDIA RTX GPUs. For this replication, we use **Detectron2**, a standard library for object detection research.

**Kaggle Note:** We install Detectron2 from source to ensure compatibility with Kaggle's pre-installed PyTorch version. We also ensure pyyaml is pinned to prevent dependency conflicts.

In [4]:
# Installation (Uncomment if needed)
# !pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu111/torch1.9/index.html

!pip install git+https://github.com/facebookresearch/detectron2.git

import os
import copy
import torch
import numpy as np
from datetime import datetime
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, default_setup
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_train_loader
from detectron2.data import transforms as T
from detectron2.data import detection_utils as utils
from detectron2 import model_zoo

print(f"Using Torch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-cbumz3bp
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-cbumz3bp
  Resolved https://github.com/facebookresearch/detectron2.git to commit fd27788985af0f4ca800bca563acdb700bb890e2
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 7.2 MB/s eta 0:00:00
  Created wheel for detectron2: filename=detectron2-0.6-cp312-cp312-linux_x86_64.whl size=6733228 sha256=69c824a4a3a21b0ae6d823a58b9c914b03765bab971fab96376cb029b39873e6
  Stored in directory: /tmp/pip-ephem-wheel-cache-8_tx_z52/wheels/d3/6e/bd/1969578f1456a6be2d6f083da65c669f450b23b8f3d1ac14c1
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=808e25f23c17a0ccdfe41b197

2\. Dataset Registration
------------------------

As per **Section 3.1**, the dataset consists of 5,900 images of paprika plants. We follow the paper's split:

*   **70% Training**
    
*   **10% Validation**
    
*   **20% Testing**
    

The DDL unit focuses on 6 abnormality categories found in the Paprika dataset.

In [ ]:
# 6 Classes from the dataset README
# Note: Ensure this order matches the ID (0-5) in the YOLO .txt files.
# Usually YOLO starts at 0. Check your _classes.txt if avail.
CLASS_NAMES = [
    "blossom_end_rot",
    "graymold",
    "powdery_mildew",
    "spider_mite",
    "spotting_disease",
    "snails_and_slugs"
]

# === ADJUST THIS PATH ===
# Based on your screenshot, it is likely:
# /kaggle/input/paprika-dataset/data
# OR /kaggle/input/{your-dataset-name}/data
DATASET_ROOT = "/kaggle/input/paprika-dataset/data" 

def get_paprika_dicts(img_dir, label_dir):
    """
    Parses YOLO format dataset for Detectron2.
    """
    dataset_dicts = []
    # Find all images (support jpg and png)
    image_files = glob.glob(os.path.join(img_dir, "*.jpg")) + glob.glob(os.path.join(img_dir, "*.png"))
    
    print(f"Found {len(image_files)} images in {img_dir}")
    
    for idx, img_path in enumerate(image_files):
        record = {}
        
        # 1. Get Image Dimensions (Needed for YOLO relative -> absolute conversion)
        # We use PIL to lazy load just the size (faster than reading full image)
        with Image.open(img_path) as img:
            width, height = img.size
            
        record["file_name"] = img_path
        record["image_id"] = idx
        record["height"] = height
        record["width"] = width
        
        # 2. Find Corresponding Label File
        # YOLO structure: images/file.jpg -> labels/file.txt
        filename = os.path.basename(img_path)
        label_filename = os.path.splitext(filename)[0] + ".txt"
        label_path = os.path.join(label_dir, label_filename)
        
        objs = []
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                lines = f.readlines()
            
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5: continue
                
                # YOLO: class_id, center_x, center_y, width, height (Normalized)
                class_id = int(parts[0])
                cx, cy, w, h = map(float, parts[1:5])
                
                # Conversion to Absolute XYXY
                abs_cx = cx * width
                abs_cy = cy * height
                abs_w = w * width
                abs_h = h * height
                
                x_min = abs_cx - (abs_w / 2)
                y_min = abs_cy - (abs_h / 2)
                x_max = abs_cx + (abs_w / 2)
                y_max = abs_cy + (abs_h / 2)
                
                obj = {
                    "bbox": [x_min, y_min, x_max, y_max],
                    "bbox_mode": BoxMode.XYXY_ABS,
                    "category_id": class_id,
                }
                objs.append(obj)
        
        record["annotations"] = objs
        dataset_dicts.append(record)
        
    return dataset_dicts

def register_paprika_datasets():
    # Register Train and Valid based on folder structure
    splits = ["train", "valid"]
    
    for split in splits:
        name = f"paprika_{split}"
        img_dir = os.path.join(DATASET_ROOT, split, "images")
        label_dir = os.path.join(DATASET_ROOT, split, "labels")
        
        # Only register if path exists
        if os.path.exists(img_dir):
            DatasetCatalog.register(name, lambda d=img_dir, l=label_dir: get_paprika_dicts(d, l))
            MetadataCatalog.get(name).set(thing_classes=CLASS_NAMES)
            print(f"Successfully Registered: {name}")
        else:
            print(f"Warning: Directory not found: {img_dir}")

# Clear previous registrations if re-running
DatasetCatalog.clear()
register_paprika_datasets()

# Verification
if "paprika_train" in DatasetCatalog.list():
    dataset_dicts = DatasetCatalog.get("paprika_train")
    if len(dataset_dicts) > 0:
        d = dataset_dicts[0]
        img = utils.read_image(d["file_name"], format="BGR")
        visualizer = Visualizer(img[:, :, ::-1], metadata=MetadataCatalog.get("paprika_train"), scale=0.5)
        out = visualizer.draw_dataset_dict(d)
        plt.figure(figsize=(10, 10))
        plt.imshow(out.get_image()[:, :, ::-1])
        plt.title("Sample Data Verification")
        plt.show()
    else:
        print("Dataset registered but no images found.")
